# Lesson 9 — Model-based reinforcement learning and the two-step task

Everything so far has been **model-free**: the agent cached a value for each option and nudged it with prediction errors. But people also *plan*. Tell a participant that the vending machine's buttons have been rewired and they adjust at once, with no trial-and-error; a value-caching learner would have to re-learn every button. This lesson introduces the second great family of RL models — **model-based** control, which computes values by reasoning over a model of the task — and the experiment that has dominated the empirical study of the two systems in humans, the **two-step task** of Daw et al. (2011). You will build the planner, simulate the hybrid model that mixes the two systems, read the behavioural signature that separates them, and fit the model to data.

**What you will learn**

- the difference between caching values (model-free) and computing them from a model (model-based), and why both exist,
- how to plan with a known model by **value iteration**,
- the design and logic of the two-step task and its **stay-probability** signature,
- how to implement, simulate and fit the **hybrid model** of Daw et al. (2011) with its model-based weight $w$,
- what can go wrong when interpreting $w$ (and which parameterisations the field uses).

**Prerequisites:** lessons 3, 4 and 5 (Rescorla–Wagner and softmax, TD learning, maximum likelihood).

**Estimated time:** 2.5 hours.

In [ ]:
import sys; sys.path.insert(0, "..")      # find the rlcourse package when running from notebooks/
import numpy as np
import matplotlib.pyplot as plt
from rlcourse import check, hint, reveal, progress, use, setup_style
from rlcourse import models, tasks, fitting, plotting
setup_style()
rng = np.random.default_rng(2024)        # one seed per notebook: results are reproducible

## 9.1 Habits and goals: two ways to compute a value

The algorithms of lessons 3 and 4 learn **cached values**: a number per state or action, updated by reward prediction errors. Such a learner is *model-free* — it stores no knowledge of what leads to what, only of how good things turned out. The great advantage is cheapness: choosing is a table look-up. The great weakness is inflexibility: when the world changes, every cached value that depends on the change must be re-learned through experience.

A **model-based** learner stores a *model* of the task instead — the transition structure $P(s' \mid s, a)$ (what follows what) and the rewards $R(s')$ — and computes values on demand by **planning**: mentally simulating the consequences of each action through the model. This is more expensive (planning takes time and working memory) but immediately flexible: change the model and the values change with it.

The psychology behind this distinction predates RL. Animals that have learned to press a lever for food usually stop pressing when the food is later *devalued* (for example by making them ill after eating it) — their behaviour is **goal-directed**, driven by the current value of the outcome. After extended training, however, pressing continues despite devaluation: the behaviour has become a **habit**, insensitive to the outcome. Daw, Niv and Dayan (2005) proposed that goal-directed and habitual control correspond to model-based and model-free RL, and that the brain arbitrates between them according to the uncertainty of each system's estimates (see Dolan & Dayan, 2013, for a review). Gläscher et al. (2010) showed that human brains also carry a *state prediction error* — the learning signal for the model itself — distinct from the reward prediction error of lesson 4.

| | model-free (habit) | model-based (goal-directed) |
|---|---|---|
| what is learned | cached values $Q(s,a)$ | a model: $P(s'\mid s,a)$ and $R$ |
| learning signal | reward prediction error $\delta$ | state prediction error, reward estimates |
| how a choice is made | compare stored values | plan through the model, then compare |
| cost per decision | low | high (planning) |
| after the world changes | slow re-learning | immediate adjustment |

## 9.2 Planning with a model: value iteration

Suppose the model is known. The value of a state under optimal behaviour obeys the **Bellman optimality equation**

$$
V^*(s) = \max_a \sum_{s'} P(s' \mid s, a)\,\big[R(s') + \gamma\, V^*(s')\big],
$$

read as: *the best I can do from $s$ is to pick the action whose expected immediate reward plus discounted future value is largest*. It is the same recursion TD learning approximated from samples in lesson 4 — but now, because $P$ and $R$ are known, we can solve it by brute force. **Value iteration** starts with $V = 0$ and repeatedly applies the right-hand side to every state; each sweep looks one step further into the future, and the values converge to $V^*$ (Sutton & Barto, 2018, ch. 4). Nothing is experienced: the agent reasons.

We use the convention of `tasks.MDP`: a reward $R(s')$ is received on *entering* state $s'$, and terminal states have value 0 because nothing follows them.

### ✏️ Exercise 9.1 — Planning by value iteration *(coding)*

Complete the value-iteration sweep. `P[s, a, s']` holds the transition probabilities, `R[s']` the reward on entering $s'$ and `terminal[s]` is `True` for terminal states (keep their value at 0). Return the array of state values after `n_iter` sweeps.

In [ ]:
def value_iteration(P, R, terminal, gamma, n_iter=100):
    """Optimal state values of a small MDP by value iteration (rewards R[s'] on entering s')."""
    P = np.asarray(P, dtype=float)
    R = np.asarray(R, dtype=float)
    n_states, n_actions = P.shape[0], P.shape[1]
    V = np.zeros(n_states)
    for _ in range(n_iter):
        V_new = np.zeros(n_states)
        for s in range(n_states):
            if terminal[s]:
                continue                                       # nothing follows a terminal state
            q = [... for a in range(n_actions)]                # <- one-step look-ahead for each action a
            V_new[s] = ...                                     # <- plan: assume the best action is taken
        V = V_new
    return V

check("9.1", value_iteration)

Let's plan in a small grid world with two wall cells. The planner never moves; it computes the value of every cell and the best move from each (arrows).

In [ ]:
value_iteration = use("9.1", value_iteration)
gamma = 0.9

def greedy_arrows(P, R, terminal, gamma, V):
    """The best action in each state according to the planned values."""
    Q = np.einsum("sat,t->sa", P, R + gamma * V)        # Q(s,a) = sum_s' P(s'|s,a) [R(s') + gamma V(s')]
    return np.array(["↑", "→", "↓", "←"])[np.argmax(Q, axis=1)]

def show_grid_values(grid, V, ax, title):
    arrows = greedy_arrows(grid.P, grid.R, grid.terminal, gamma, V)
    ax.imshow(V.reshape(grid.shape), cmap=plotting.SEQUENTIAL, vmin=0, vmax=1)
    n_cols = grid.shape[1]
    for s in range(grid.n_states):
        r, c = divmod(s, n_cols)
        is_wall = (not grid.terminal[s]) and np.all(grid.P[s, :, s] == 1)
        label = "goal" if grid.terminal[s] else ("wall" if is_wall else f"{V[s]:.2f}\n{arrows[s]}")
        ax.text(c, r, label, ha="center", va="center", fontsize=9, color="white" if V[s] > 0.55 else "#0b0b0b")
    ax.set(title=title, xticks=[], yticks=[])
    ax.grid(False)

grid = tasks.gridworld(n_rows=4, n_cols=4, goal=(0, 3), start=(3, 0), walls=[(1, 1), (2, 1)])
V = value_iteration(grid.P, grid.R, grid.terminal, gamma)
fig, ax = plt.subplots(figsize=(4.4, 4.2))
show_grid_values(grid, V, ax, "Planned values and best moves (γ = 0.9)")
plt.show()

Now the crucial demonstration. Move the goal to the opposite corner — a change in $R$ — and plan again. A model-free agent would have to wander around and update dozens of cached values; the planner just re-runs the computation.

In [ ]:
grid_moved = tasks.gridworld(n_rows=4, n_cols=4, goal=(3, 3), start=(3, 0), walls=[(1, 1), (2, 1)])
V_moved = value_iteration(grid_moved.P, grid_moved.R, grid_moved.terminal, gamma)
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
show_grid_values(grid, V, axes[0], "goal top-right")
show_grid_values(grid_moved, V_moved, axes[1], "goal moved to bottom-right: re-planned at once")
plt.show()

### A one-step special case

The two-step task has only two stages, so planning reduces to a single look-ahead. If $Q_2(s', a')$ are the values of the second-stage actions and $P(s' \mid a)$ the known transition probabilities, the model-based value of a first-stage action is

$$
Q_{MB}(a) = \sum_{s'} P(s' \mid a)\; \max_{a'} Q_2(s', a') .
$$

The planner assumes it will pick the best second-stage action wherever it lands, and weights each landing state by how likely action $a$ is to get there.

### ✏️ Exercise 9.2 — A model-based value by hand *(maths)*

The stage-2 values are $Q_2 = \begin{pmatrix} 0.8 & 0.2 \\ 0.4 & 0.6 \end{pmatrix}$ (rows: states 0 and 1; columns: actions 0 and 1) and the transition matrix is the usual $P(s' \mid a) = \begin{pmatrix} 0.7 & 0.3 \\ 0.3 & 0.7 \end{pmatrix}$ (rows: first-stage actions). What is $Q_{MB}$ for first-stage action 0?

In [ ]:
answer = None   # <- your number
check("9.2", answer)

### ✏️ Exercise 9.3 — Model-based stage-1 values *(coding)*

Implement the look-ahead for both first-stage actions at once. `Q2` is the 2 × 2 array of stage-2 values (rows = states, columns = actions) and `transitions[a, s']` the transition probabilities. Return an array with $Q_{MB}(0)$ and $Q_{MB}(1)$.

In [ ]:
def two_step_mb_values(Q2, transitions):
    """Model-based stage-1 values: Q_MB(a) = sum_s' P(s'|a) max_a' Q2(s', a')."""
    return ...

check("9.3", two_step_mb_values)

## 9.3 The two-step task

Daw et al. (2011) designed a task in which model-free and model-based learners behave *differently*, so that their relative influence can be read off from choices. Each trial has two stages:

1. **Stage 1.** The participant chooses between two options (say, two coloured boxes). Each option leads with probability 0.7 to its own "common" second-stage state and with probability 0.3 to the other, "rare", state.
2. **Stage 2.** In the second-stage state the participant chooses between two further options, each rewarded with its own probability. These probabilities **drift** slowly and independently (Gaussian random walks within [0.25, 0.75]), so the participant has to keep learning throughout the ~200 trials.

Participants are told the transition structure (common / rare) in advance, which is why the model can assume it is known.

In [ ]:
task = tasks.TwoStepTask(n_trials=200, rng=rng)
print("P(stage-2 state | stage-1 action):\n", task.transitions)

fig, ax = plt.subplots(figsize=(8, 3.6))
for s in range(2):
    for a in range(2):
        ax.plot(task.reward_probs[:, s, a], ls="-" if s == 0 else "--", label=f"state {s}, action {a}")
ax.set(xlabel="trial", ylabel="P(reward)", ylim=(0.2, 0.8),
       title="Stage-2 reward probabilities drift (random walks within [0.25, 0.75])")
ax.legend(ncol=2, fontsize=9)
plt.show()

### The logic of the design

Consider what each system does after a trial in which the participant chose option A at stage 1, went through a **rare** transition to the state usually reached by B, and was **rewarded** there.

- A **model-free** learner credits the action it took: A was followed by reward, so A's value goes up and the participant tends to *stay* with A. Reward raises staying regardless of the transition — a **main effect of reward**.
- A **model-based** learner knows that the rewarded state is usually reached by B. The reward raises the planned value of **B**, so the participant tends to *switch*. After a *common* rewarded transition both systems agree (stay). The result is a **reward × transition interaction**.

The standard summary is therefore the probability of repeating the stage-1 choice ("staying") as a function of the previous trial's reward and transition type:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
plotting.plot_stay_probabilities(np.array([[0.85, 0.85], [0.60, 0.60]]), ax=axes[0], title="pure model-free (idealised)")
plotting.plot_stay_probabilities(np.array([[0.85, 0.60], [0.60, 0.85]]), ax=axes[1], title="pure model-based (idealised)")
plt.show()

### ✏️ Exercise 9.4 — The behavioural signature of model-based control *(conceptual)*

Which pattern of stay probabilities indicates model-based control in the two-step task?

**(a)** Participants repeat their first-stage choice after a reward regardless of the transition type (a main effect of reward).
**(b)** Participants repeat after rewarded-common and unrewarded-rare trials, and switch after rewarded-rare and unrewarded-common trials (a reward × transition interaction).
**(c)** Participants almost always repeat their previous first-stage choice, whatever happened.
**(d)** Participants repeat more after rare transitions than after common ones, regardless of reward (a main effect of transition).

In [ ]:
check("9.4", None)   # <- replace None with "a", "b", "c" or "d"

## 9.4 The hybrid model

Daw et al. (2011) fitted a model in which both systems run side by side and their stage-1 values are mixed. Write $a_1$ for the stage-1 action, $s_2$ for the stage-2 state, $a_2$ for the stage-2 action and $r$ for the reward.

**Model-free system: SARSA(λ).** Stage-2 values $Q_2$ and model-free stage-1 values $Q_{MF}$ are updated by TD with eligibility traces (lesson 4):

$$
\delta_1 = Q_2(s_2, a_2) - Q_{MF}(a_1), \qquad Q_{MF}(a_1) \leftarrow Q_{MF}(a_1) + \alpha_1 \delta_1
$$

$$
\delta_2 = r - Q_2(s_2, a_2), \qquad Q_2(s_2, a_2) \leftarrow Q_2(s_2, a_2) + \alpha_2 \delta_2, \qquad Q_{MF}(a_1) \leftarrow Q_{MF}(a_1) + \alpha_1 \lambda\, \delta_2
$$

The first update passes the value of the state reached back to the first-stage action; the $\lambda$-weighted update passes the final reward back directly (the eligibility trace).

**Model-based system.** Uses the same $Q_2$ and the known transitions: $Q_{MB}(a) = \sum_{s'} P(s'\mid a) \max_{a'} Q_2(s', a')$ — Exercise 9.3.

**Mixing and choice.** The two stage-1 values are combined with a weight $w \in [0, 1]$,

$$
Q_{net}(a) = w\, Q_{MB}(a) + (1 - w)\, Q_{MF}(a),
$$

and choices are softmax with a **perseveration** bonus $p$ for the action chosen on the previous trial ($\mathrm{rep}(a) = 1$ if $a$ was chosen last time, else 0):

$$
P(a_1 = a) \propto \exp\!\big(\beta_1 [Q_{net}(a) + p \cdot \mathrm{rep}(a)]\big), \qquad P(a_2 = a) \propto \exp\!\big(\beta_2\, Q_2(s_2, a)\big).
$$

Seven parameters: $\alpha_1, \alpha_2, \lambda, \beta_1, \beta_2, w, p$. In `rlcourse` they are always passed in this order (`models.TWO_STEP_PARAM_NAMES`). $w = 1$ is a pure planner, $w = 0$ a pure habit learner.

In [ ]:
from rlcourse.models import softmax, choose      # the choice helpers from lesson 3
print(models.TWO_STEP_PARAM_NAMES)
print(models.TWO_STEP_PARAM_KINDS)                # the parameter transforms used when fitting (lesson 5)

### ✏️ Exercise 9.5 — Simulate the hybrid agent *(coding)*

Complete the trial loop. Use `task.step1(a1)` (returns the stage-2 state and whether the transition was common) and `task.step2(a2)` (returns the reward and advances the task). Record `a1`, `s2`, `a2`, `r` and `common` (1 for common) for every trial.

The checker does not require the same random numbers as the reference: it verifies that `s2` is consistent with `a1` and `common`, and that your agent shows the model-based interaction with $w = 1$ and the model-free main effect with $w = 0$.

In [ ]:
def two_step_simulate(params, task, rng):
    """Simulate the Daw et al. (2011) hybrid agent on a TwoStepTask.

    params = (alpha1, alpha2, lam, beta1, beta2, w, p).  Returns a dict of arrays a1, s2, a2, r, common.
    """
    alpha1, alpha2, lam, beta1, beta2, w, p_stick = params
    n = task.n_trials
    Q_mf = np.zeros(2)                 # model-free stage-1 values
    Q2 = np.zeros((2, 2))              # stage-2 values (shared by both systems)
    a1 = np.zeros(n, int); s2 = np.zeros(n, int); a2 = np.zeros(n, int)
    r = np.zeros(n); common = np.zeros(n, int)
    prev_a1 = None
    task.reset()
    for t in range(n):
        raise NotImplementedError("your code here: stage-1 choice, transition, stage-2 choice, reward, learning")
    return {"a1": a1, "s2": s2, "a2": a2, "r": r, "common": common}

check("9.5", two_step_simulate)

In [ ]:
import pandas as pd
two_step_simulate = use("9.5", two_step_simulate)

example_task = tasks.TwoStepTask(n_trials=300, rng=rng)
example_data = two_step_simulate([0.5, 0.5, 0.6, 5.0, 5.0, 0.7, 0.1], example_task, rng)
pd.DataFrame(example_data).head(8)        # the first eight trials of one simulated participant

### ✏️ Exercise 9.6 — Stay probabilities *(coding)*

Compute the 2 × 2 matrix of stay probabilities: rows = previous trial rewarded / unrewarded, columns = previous transition common / rare. A "stay" on trial $t$ means the stage-1 choice equals that of trial $t-1$, and it is classified by what happened on trial $t-1$.

In [ ]:
def stay_probabilities(data):
    """P(repeat the stage-1 choice) split by the previous trial's reward (rows: rewarded, unrewarded)
    and transition (columns: common, rare)."""
    a1 = np.asarray(data["a1"]); r = np.asarray(data["r"]); common = np.asarray(data["common"])
    stay = ...                                   # <- 1 if a1[t] == a1[t-1], for t = 1 .. n-1
    prev_r, prev_common = ..., ...               # <- the reward and transition of the *previous* trial
    out = np.zeros((2, 2))
    for i, rewarded in enumerate([1, 0]):
        for j, is_common in enumerate([1, 0]):
            mask = ...                           # <- trials whose previous outcome matches (rewarded, is_common)
            out[i, j] = stay[mask].mean() if mask.any() else np.nan
    return out

check("9.6", stay_probabilities)

Now the key simulation: the same agent with $w = 0$, $0.5$ and $1$.

In [ ]:
stay_probabilities = use("9.6", stay_probabilities)

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), sharey=True)
for ax, w in zip(axes, [0.0, 0.5, 1.0]):
    params = [0.5, 0.5, 0.6, 6.0, 6.0, w, 0.1]            # alpha1, alpha2, lam, beta1, beta2, w, p
    sim_task = tasks.TwoStepTask(n_trials=600, rng=rng)
    sim_data = two_step_simulate(params, sim_task, rng)
    plotting.plot_stay_probabilities(stay_probabilities(sim_data), ax=ax, title=f"simulated agent, w = {w}", ylim=(0.3, 1.0))
plt.show()

The pure model-free agent shows the main effect of reward, the pure planner the crossover interaction, and the mixed agent a blend of the two — the pattern typically seen in human participants (Daw et al., 2011, Figure 2).

### ✏️ Exercise 9.7 — Perseveration can flip a choice *(maths)*

On the current trial $Q_{net}(A) = 0.5$ and $Q_{net}(B) = 0.4$. On the previous trial the participant chose **B**. With $p = 0.2$ and $\beta_1 = 3$, what is the probability of choosing A? Give three decimals.

In [ ]:
answer = None   # <- your number
check("9.7", answer)

## 9.5 Likelihood and fitting

To fit the model we turn the simulation inside out, exactly as in lesson 5: run the model forward on the participant's **observed** sequence of $(a_1, s_2, a_2, r)$, and on every trial accumulate the log-probability the model assigned to the choices that were actually made — at **both** stages.

### ✏️ Exercise 9.8 — The likelihood of two-step data *(coding)*

Write the negative log-likelihood. The order inside a trial matters: compute the stage-1 probabilities and score $a_1$, compute the stage-2 probabilities and score $a_2$, *then* learn from $(s_2, a_2, r)$ and remember $a_1$ for the next trial's perseveration.

In [ ]:
def two_step_nll(params, data, transitions=None):
    """Negative log-likelihood of two-step data under the hybrid model.

    params = (alpha1, alpha2, lam, beta1, beta2, w, p); data as returned by two_step_simulate."""
    alpha1, alpha2, lam, beta1, beta2, w, p_stick = params
    if transitions is None:
        transitions = np.array([[0.7, 0.3], [0.3, 0.7]])
    Q_mf = np.zeros(2)
    Q2 = np.zeros((2, 2))
    prev_a1 = None
    nll = 0.0
    for a1, s2, a2, r in zip(data["a1"], data["s2"], data["a2"], data["r"]):
        raise NotImplementedError("your code here: probabilities of a1 and a2, then the three learning updates")
    return nll

check("9.8", two_step_nll)

Fit one simulated participant with 300 trials. `fitting.fit_mle` (lesson 5) optimises in unconstrained space using the transforms in `models.TWO_STEP_PARAM_KINDS` (learning rates and $w$ through a sigmoid, inverse temperatures through an exponential, $p$ unconstrained).

In [ ]:
two_step_nll = use("9.8", two_step_nll)

true_params = [0.5, 0.4, 0.6, 5.0, 4.0, 0.6, 0.1]
fit_task = tasks.TwoStepTask(n_trials=300, rng=rng)
fit_data = two_step_simulate(true_params, fit_task, rng)
fit = fitting.fit_mle(two_step_nll, (fit_data,), models.TWO_STEP_PARAM_KINDS, n_starts=3, rng=rng)

print(f"negative log-likelihood at the fit: {fit['nll']:.1f}   (at the true parameters: {two_step_nll(true_params, fit_data):.1f})")
pd.DataFrame({"true": true_params, "fitted": np.round(fit["params"], 2)}, index=models.TWO_STEP_PARAM_NAMES)

Some parameters come back close, others not — with seven parameters and 300 noisy trials this is normal. The one most studies care about is $w$, so let's check how well *it* is recovered (lesson 6) across a dozen simulated participants.

In [ ]:
n_subjects, n_trials = 12, 200
true_w = rng.uniform(0.05, 0.95, n_subjects)
recovered_w = np.zeros(n_subjects)
for i in range(n_subjects):
    params = [0.5, 0.5, 0.6, 5.0, 5.0, true_w[i], 0.1]
    sub_task = tasks.TwoStepTask(n_trials=n_trials, rng=rng)
    sub_data = two_step_simulate(params, sub_task, rng)
    sub_fit = fitting.fit_mle(two_step_nll, (sub_data,), models.TWO_STEP_PARAM_KINDS, n_starts=2, rng=rng)
    recovered_w[i] = sub_fit["params"][5]
plotting.plot_parameter_recovery(true_w[:, None], recovered_w[:, None], ["w"])
plt.show()

With 200 trials and all other parameters free, $w$ is recovered only roughly: the fit must disentangle it from $\beta_1$ and $\lambda$, which shape the same stay probabilities. This is why two-step studies use many trials (Daw et al. used 201 trials per session), fix or constrain nuisance parameters, or estimate parameters hierarchically (lesson 8) — and why lesson 14 insists on checking recovery in your own design before interpreting individual differences in $w$.

### ✏️ Exercise 9.9 — Which parameter indexes model-based control? *(conceptual)*

In the hybrid model of Daw et al. (2011), the degree of model-based control is measured by ...

**(a)** $\alpha_1$, the stage-1 learning rate
**(b)** $\beta_1$, the stage-1 inverse temperature
**(c)** $w$, the weight of the model-based values in the stage-1 decision
**(d)** $\lambda$, the eligibility-trace parameter

In [ ]:
check("9.9", None)   # <- replace None with "a", "b", "c" or "d"

### A note on parameterisations

Many later studies — including the large online study of compulsivity by Gillan et al. (2016), which lesson 14 revisits — use an algebraically equivalent reparameterisation: instead of one inverse temperature $\beta_1$ and a weight $w$, each system gets its own inverse temperature, $\beta_{MB}$ and $\beta_{MF}$ (roughly $\beta_{MB} \approx w\beta_1$ and $\beta_{MF} \approx (1-w)\beta_1$), and the eligibility trace is replaced by separate TD(0) and TD(1) model-free learners. The advantage is that the *strength* of model-based control becomes a single parameter that does not interact with an overall temperature. When you read a paper, check which parameterisation it uses before comparing numbers.

## 9.6 Caveats and current debates

The two-step task is a workhorse, but its interpretation has been refined considerably since 2011:

- **Confounds in the signature.** Akam, Costa and Dayan (2015) showed that simple strategies that are not model-based (for example tracking which second-stage state was recently rewarding) can mimic the interaction, and that details of the task statistics matter. Model fitting is less vulnerable than the stay-probability regression, but not immune.
- **Is planning worth it?** In the original task the drift is slow and the stakes are small; Kool, Cushman and Gershman (2016) showed that model-based control barely improves earnings in it — which may explain why many participants look mostly model-free. Variants with larger reward differences and deterministic transitions change this.
- **Resources.** Model-based control draws on working memory and attention: it drops under cognitive load and stress (Otto et al., 2013), and its depth may be limited and adaptive (Keramati et al., 2016).
- **Arbitration.** The fixed weight $w$ is a simplification. Lee, Shimojo and O'Doherty (2014) modelled trial-by-trial arbitration based on the reliability of each system's predictions, with neural correlates in the prefrontal cortex.
- **Knowing the model.** The hybrid model assumes the transition structure is known and used perfectly. Learning the transitions itself can be impaired — for example, Sharp, Dolan and Eldar (2023) relate disrupted state-transition learning to compulsivity — so a low $w$ can reflect a poor model rather than a lack of planning.

### ✏️ Exercise 9.10 — Why rare transitions are diagnostic *(conceptual, written)*

In 2–4 sentences: why are trials with a *rare* transition the ones that reveal whether a participant plans? Write your answer in the string below.

In [ ]:
check("9.10", """
    Write your answer here.
""")

## Summary

- **Model-free** control caches values and updates them with prediction errors; **model-based** control learns a model (transitions and rewards) and computes values by planning. The first is cheap but inflexible, the second flexible but costly.
- **Value iteration** solves the Bellman optimality equation by repeated one-step look-ahead; in a two-stage task it reduces to $Q_{MB}(a) = \sum_{s'} P(s'\mid a)\max_{a'} Q_2(s', a')$.
- In the **two-step task** the two systems disagree only after rare transitions: a model-free learner shows a main effect of reward on staying, a model-based learner a reward × transition interaction.
- The **hybrid model** of Daw et al. (2011) runs SARSA(λ) and a planner on shared stage-2 values and mixes them with $w$; it is simulated and fitted with exactly the machinery of lessons 3–5.
- $w$ is recoverable only with enough trials and care; alternative parameterisations and several caveats (confounds, incentives, resources, arbitration, model learning) matter for interpretation.

## What's next

Lesson 10 introduces a third kind of representation that sits between the two systems — the **successor representation**, which caches *predictions about future states* rather than values or a full model — and uses revaluation experiments to show exactly what it can and cannot adapt to.

## References

- Daw, N. D., Niv, Y., & Dayan, P. (2005). Uncertainty-based competition between prefrontal and dorsolateral striatal systems for behavioral control. *Nature Neuroscience, 8*, 1704–1711. https://doi.org/10.1038/nn1560
- Gläscher, J., Daw, N., Dayan, P., & O'Doherty, J. P. (2010). States versus rewards: Dissociable neural prediction error signals underlying model-based and model-free reinforcement learning. *Neuron, 66*(4), 585–595. https://doi.org/10.1016/j.neuron.2010.04.016
- Daw, N. D., Gershman, S. J., Seymour, B., Dayan, P., & Dolan, R. J. (2011). Model-based influences on humans' choices and striatal prediction errors. *Neuron, 69*(6), 1204–1215. https://doi.org/10.1016/j.neuron.2011.02.027
- Dolan, R. J., & Dayan, P. (2013). Goals and habits in the brain. *Neuron, 80*(2), 312–325. https://doi.org/10.1016/j.neuron.2013.09.007
- Otto, A. R., Raio, C. M., Chiang, A., Phelps, E. A., & Daw, N. D. (2013). Working-memory capacity protects model-based learning from stress. *PNAS, 110*(52), 20941–20946. https://doi.org/10.1073/pnas.1312011110
- Lee, S. W., Shimojo, S., & O'Doherty, J. P. (2014). Neural computations underlying arbitration between model-based and model-free learning. *Neuron, 81*(3), 687–699. https://doi.org/10.1016/j.neuron.2013.11.028
- Akam, T., Costa, R., & Dayan, P. (2015). Simple plans or sophisticated habits? State, transition and learning interactions in the two-step task. *PLoS Computational Biology, 11*(12), e1004648. https://doi.org/10.1371/journal.pcbi.1004648
- Kool, W., Cushman, F. A., & Gershman, S. J. (2016). When does model-based control pay off? *PLoS Computational Biology, 12*(8), e1005090. https://doi.org/10.1371/journal.pcbi.1005090
- Gillan, C. M., Kosinski, M., Whelan, R., Phelps, E. A., & Daw, N. D. (2016). Characterizing a psychiatric symptom dimension related to deficits in goal-directed control. *eLife, 5*, e11305. https://doi.org/10.7554/eLife.11305
- Keramati, M., Smittenaar, P., Dolan, R. J., & Dayan, P. (2016). Adaptive integration of habits into depth-limited planning defines a habitual-goal-directed spectrum. *PNAS, 113*(45), 12868–12873. https://doi.org/10.1073/pnas.1609094113
- Sharp, P. B., Dolan, R. J., & Eldar, E. (2023). Disrupted state transition learning as a computational marker of compulsivity. *Psychological Medicine, 53*(5), 2095–2105. https://doi.org/10.1017/S0033291721003846
- Sutton, R. S. (1990). Integrated architectures for learning, planning, and reacting based on approximating dynamic programming. *Proceedings of the Seventh International Conference on Machine Learning*, 216–224.
- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.), chapters 4 and 8. MIT Press. http://incompleteideas.net/book/the-book-2nd.html